In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# Chemin absolu vers la racine du projet
BASE = r"C:\Users\izouk\OneDrive\Desktop\projet aeroport\airline_performance"

con = duckdb.connect()

# Chargement des marts avec chemins absolus
df_merger  = con.execute(f"SELECT * FROM read_parquet('{BASE}/data/gold/mart_merger_impact.parquet')").df()
df_monthly = con.execute(f"SELECT * FROM read_parquet('{BASE}/data/gold/mart_otp_monthly.parquet')").df()

print("mart_merger_impact :", df_merger.shape)
print("mart_otp_monthly   :", df_monthly.shape)
print(df_merger.head())

mart_merger_impact : (178, 17)
mart_otp_monthly   : (1798, 12)
  carrier_key  YEAR  QUARTER fusion_periode   carrier_group  total_flights  \
0          AA  2009        1            pre  American Group         138473   
1          AA  2009        2            pre  American Group         138367   
2          AA  2009        3            pre  American Group         183414   
3          AA  2009        4            pre  American Group          89802   
4          AA  2010        1            pre  American Group         136174   

   otp_pct  avg_arr_delay  cancellation_rate  min_carrier  min_late_aircraft  \
0    80.37          33.27               2.66     525415.0           457530.0   
1    75.76          38.41               1.91     640212.0           693916.0   
2    81.17          32.68               0.88     737804.0           647550.0   
3    81.63          32.12               1.41     328696.0           285791.0   
4    79.15          32.40               2.94     546422.0           

In [4]:
carriers_fusion = ['UA', 'CO', 'WN', 'FL', 'AA', 'US']

df_line = df_merger[df_merger['carrier_key'].isin(carriers_fusion)].copy()
df_line['period'] = df_line['YEAR'].astype(str) + '-Q' + df_line['QUARTER'].astype(str)

fig = px.line(
    df_line,
    x='period',
    y='otp_pct',
    color='carrier_key',
    markers=True,
    title='OTP trimestriel — carriers fusionnés (2009-2018)',
    labels={'otp_pct': 'OTP %', 'period': 'Trimestre', 'carrier_key': 'Carrier'}
)

# Annotations fusions (sans add_vline car axe catégoriel)
fusions = {
    '2010-Q4': 'UA+CO',
    '2011-Q2': 'WN+FL',
    '2013-Q4': 'AA+US'
}

for periode, label in fusions.items():
    fig.add_annotation(
        x=periode,
        y=100,
        text=label,
        showarrow=True,
        arrowhead=2,
        arrowcolor='gray',
        font=dict(color='gray', size=11),
        ax=0,
        ay=-30
    )

fig.update_layout(height=500, xaxis_tickangle=45)
fig.show()

In [5]:
ordre = ['pre', 'during', 'post']

df_avg = df_merger[
    df_merger['carrier_key'].isin(carriers_fusion)
].groupby(['carrier_key', 'fusion_periode'])['otp_pct'].mean().reset_index()

df_avg['fusion_periode'] = pd.Categorical(df_avg['fusion_periode'], categories=ordre, ordered=True)
df_avg = df_avg.sort_values('fusion_periode')

fig = px.bar(
    df_avg,
    x='carrier_key',
    y='otp_pct',
    color='fusion_periode',
    barmode='group',
    title='OTP moyen — avant / pendant / après fusion',
    labels={'otp_pct': 'OTP %', 'carrier_key': 'Carrier', 'fusion_periode': 'Période'},
    color_discrete_map={'pre': '#2196F3', 'during': '#FF5722', 'post': '#4CAF50'}
)

fig.update_layout(height=450)
fig.show()

In [6]:
fig = px.line(
    df_merger[df_merger['carrier_key'].isin(carriers_fusion)],
    x='YEAR',
    y='pct_late_aircraft',
    color='carrier_key',
    markers=True,
    title='Part LATE_AIRCRAFT par année — effet domino post-fusion ?',
    labels={'pct_late_aircraft': '% Late Aircraft', 'YEAR': 'Année'}
)

for annee, label in [(2010, 'UA+CO'), (2011, 'WN+FL'), (2013, 'AA+US')]:
    fig.add_vline(x=annee, line_dash='dot', line_color='gray', annotation_text=label)

fig.update_layout(height=450)
fig.show()

In [9]:
parquet_path = BASE.replace("\\", "/") + "/data/parquet/*.parquet"

carriers_par_annee = con.execute(f"""
    SELECT
        YEAR,
        COUNT(DISTINCT OP_UNIQUE_CARRIER) AS nb_carriers
    FROM read_parquet('{parquet_path}')
    GROUP BY 1
    ORDER BY 1
""").df()

fig = px.bar(
    carriers_par_annee,
    x='YEAR',
    y='nb_carriers',
    title='Nombre de carriers actifs par année — consolidation sectorielle',
    labels={'nb_carriers': 'Nombre de carriers', 'YEAR': 'Année'},
    color='nb_carriers',
    color_continuous_scale='Blues'
)

fig.update_layout(height=400)
fig.show()

In [10]:
parquet_path = BASE.replace("\\", "/") + "/data/parquet/*.parquet"

vols = con.execute(f"""
    SELECT
        YEAR,
        OP_UNIQUE_CARRIER,
        COUNT(*) AS nb_vols
    FROM read_parquet('{parquet_path}')
    GROUP BY 1, 2
""").df()

total = vols.groupby('YEAR')['nb_vols'].sum().reset_index().rename(columns={'nb_vols': 'total'})
vols  = vols.merge(total, on='YEAR')
vols['share'] = vols['nb_vols'] / vols['total']
hhi   = vols.groupby('YEAR').apply(lambda x: (x['share']**2).sum()).reset_index(name='hhi')

fig = px.line(
    hhi,
    x='YEAR',
    y='hhi',
    markers=True,
    title='Indice HHI — concentration du marché 2009-2018',
    labels={'hhi': 'HHI', 'YEAR': 'Année'}
)

fig.add_hline(y=0.25, line_dash='dash', line_color='red',
              annotation_text='Seuil marché concentré')

fig.update_layout(height=400)
fig.show()

C:\Users\izouk\AppData\Local\Temp\ipykernel_33032\4093382564.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hhi   = vols.groupby('YEAR').apply(lambda x: (x['share']**2).sum()).reset_index(name='hhi')
